In [0]:
-- Relationship Integrity Check: Ensures every item in Silver actually belongs to an order in Silver
SELECT COUNT(i.order_id) AS orphan_item_count
FROM olist_maas_pipeline.silver.sales_order_items i
LEFT JOIN olist_maas_pipeline.silver.sales_orders o 
    ON i.order_id = o.order_id
WHERE o.order_id IS NULL;
-- EXPECTED RESULT: 0

In [0]:
-- Multi-Seller Proof: Confirm that our sales_order_items table correctly preserved the individual seller identities
SELECT order_id
    , COUNT(DISTINCT seller_id) as seller_count
FROM olist_maas_pipeline.silver.sales_order_items
GROUP BY order_id
HAVING seller_count > 1;
-- EXPECTED RESULT: Approximately 1,000 rows

In [0]:
-- Delivery logic check: Confirm there is no delivery happened before purchase
SELECT COUNT(*) AS illogical_date_count
FROM olist_maas_pipeline.silver.sales_orders
WHERE order_delivered_customer_date < order_purchase_timestamp;
-- EXPECTED RESULT: 0

In [0]:
-- Tier completeness check: There is no missing tier for customer or seller state
SELECT customer_state
    , COUNT(*) as missing_tier_count
FROM olist_maas_pipeline.silver.sales_orders
WHERE logistics_tier IS NULL
GROUP BY 1;
-- EXPECTED RESULT: 0

In [0]:
-- Geolocation check
SELECT geolocation_zip_code_prefix, latitude, longitude, state_code
FROM olist_maas_pipeline.silver.sales_geolocation
WHERE (latitude > 6 OR latitude < -35)
   OR (longitude > -33 OR longitude < -75)
   OR latitude IS NULL 
   OR longitude IS NULL;